# Catch metric inflation in a compliance agent with oathgate

In this cookbook, you build a compliance agent with Mistral tool calling, score it with a pass rate (code plus LLM judge), and use oathgate to notice when someone changes the rubric or the dataset to make the score look better.

*Author: Uladzimir Khadakouski ([@UladzKha](https://github.com/UladzKha)), Independent*

> **Note:** A full run of this notebook costs about €0.07 in Mistral API calls and takes about 11 minutes.

When you evaluate an agent, the score can go up even when the agent doesn't get better. You can relax the judge rubric, or remove the hard cases from the dataset. In both cases the agent's answers are the same, but the score looks better. In this cookbook, you freeze the scoring setup (the rubric, the judge, the scoring code, and the data) with oathgate before you change the prompt, so any later change to the rubric or the dataset makes `oathgate check` fail.

To evaluate and optimize a prompt with the Evaluation SDK, see the [offline evaluation cookbook](https://github.com/mistralai/cookbook/blob/main/mistral/observability/offline_evaluation_and_optimization.ipynb). This cookbook uses only the chat API, so it doesn't need the Evaluation SDK or Enterprise access. It focuses on a different question: does the score go up because the agent got better, or because the scorer changed?

## Prerequisites

To complete this cookbook, you will need:

- A Mistral API key
- Python 3.11 or later
- Google Colab or a local Jupyter environment

## Environment setup

Before you start, install the packages and set up your Mistral API key.

### Install

Install the Mistral Python SDK and oathgate. The versions are pinned so that you get the same behavior as in this notebook.

In [1]:
%pip install -q mistralai==3.8.0 oathgate==0.1.0

Note: you may need to restart the kernel to use updated packages.


### Required environment variables

To complete this cookbook, you'll need a Mistral API key. In [Studio](https://console.mistral.ai), navigate to the [API keys section](https://console.mistral.ai/api-keys) and create a new API key.

The next cell reads the key from the `MISTRAL_API_KEY` environment variable. If the variable is not set, it asks for the key with `getpass`, so the key never appears in the notebook output.

The client also retries when the API returns a rate limit error (`429`) or a server error, or when the connection times out. It waits longer after each try and stops after 2 minutes. These errors aren't the agent's fault, so they never count as an agent fail. If the retries run out, the run stops.

In [2]:
import os
from getpass import getpass
from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

RETRY = RetryConfig("backoff", BackoffStrategy(1000, 30000, 2.0, 120000), True)

api_key = os.environ.get("MISTRAL_API_KEY", "").strip()
if not api_key:
    api_key = getpass("Mistral API key: ").strip()

client = Mistral(api_key=api_key, retry_config=RETRY)
print("Mistral client created, key provided:", bool(api_key))

Mistral client created, key provided: True


In [3]:
import json
import random
from datetime import date, timedelta
from pathlib import Path
import re
import unicodedata
import difflib
from collections import Counter
import hashlib
from types import SimpleNamespace
import shutil
import importlib
import subprocess

## Build a synthetic world

In this section, you create a fake sanctions list with 12 names and 20 customers with 60 days of transactions, and save them to `world.json`. `get_customer_activity` reads the customers from this file, and `screen_name` uses the same sanctions list.

In [4]:
DATA_DIR = Path("compliance_eval/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
WORLD_PATH = DATA_DIR / "world.json"
AS_OF = date(2026, 11, 1)
HISTORY_DAYS = 60
BLOCK_THRESHOLD = 0.92
ESCALATE_THRESHOLD = 0.80

print(f"data dir: {DATA_DIR}, world_path: {WORLD_PATH}, as of: {AS_OF}, history start: {AS_OF-timedelta(days=HISTORY_DAYS)}")

data dir: compliance_eval/data, world_path: compliance_eval/data/world.json, as of: 2026-11-01, history start: 2026-09-02


In [5]:
SANCTIONS = [
    {
        "id": "SAN-001",
        "name": "Aldona Qureshi",
        "aliases": ["Aldona Kureshi"],
        "type": "individual",
        "linked_to": ["SAN-002"],
    },
    {
        "id": "SAN-002",
        "name": "Mossfjord Shipping LTD",
        "aliases": ["Mossfjord Shipping", "Mossfjord Maritime"],
        "type": "entity",
        "linked_to": ["SAN-001"],
    },
    {
        "id": "SAN-003",
        "name": "Mirela Söderqvist",
        "aliases": ["Mirela Soderqvist", "Mirela Soederqvist"],
        "type": "individual",
        "linked_to": ["SAN-004"],
    },
    {
        "id": "SAN-004",
        "name": "Veldhollow Marine GmbH",
        "aliases": ["Veldhollow Marine", "Veldhollow Maritime"],
        "type": "entity",
        "linked_to": ["SAN-003"],
    },
    {
        "id": "SAN-005",
        "name": "Tenzin Välimaa",
        "aliases": ["Tenzin Valimaa", "Tenzin Vaelimaa"],
        "type": "individual",
        "linked_to": ["SAN-006"],
    },
    {
        "id": "SAN-006",
        "name": "Cindervale Materials AG",
        "aliases": ["Cindervale Materials", "Cindervale Industrial"],
        "type": "entity",
        "linked_to": ["SAN-005"],
    },
    {
        "id": "SAN-007",
        "name": "Ivo Nakamberi",
        "aliases": ["Ivo Nacamberi"],
        "type": "individual",
        "linked_to": ["SAN-008"],
    },
    {
        "id": "SAN-008",
        "name": "Westfen Logistics SA",
        "aliases": ["Westfen Logistics", "Westfen Freight"],
        "type": "entity",
        "linked_to": ["SAN-007"],
    },
    {
        "id": "SAN-009",
        "name": "Zorán El-Märi",
        "aliases": ["Zoran El-Mari", "Zoran El Maeri"],
        "type": "individual",
        "linked_to": ["SAN-010"],
    },
    {
        "id": "SAN-010",
        "name": "Brackenfjord Analytics LLC",
        "aliases": ["Brackenfjord Analytics", "Brackenfjord Data"],
        "type": "entity",
        "linked_to": ["SAN-009"],
    },
    {
        "id": "SAN-011",
        "name": "Esmé Kowalenga",
        "aliases": [],
        "type": "individual",
        "linked_to": ["SAN-012"],
    },
    {
        "id": "SAN-012",
        "name": "Sablebrook Transit LTD",
        "aliases": ["Sablebrook Transit", "Sablebrook Transport"],
        "type": "entity",
        "linked_to": ["SAN-011"],
    },
]

In [6]:
individual = 0
entity = 0

for item in SANCTIONS:
    if item["type"] == "entity":
        entity += 1
    elif item["type"] == "individual":
        individual += 1
    else:
        print(f"Unknown sanctions record type: {item['id']} {item['type']}")

ids_unique = len(set(item["id"] for item in SANCTIONS)) == len(SANCTIONS)
print(f"total: {len(SANCTIONS)}",
      f"individuals: {individual}",
      f"entity: {entity}",
      f"ids unique: {ids_unique}")

total: 12 individuals: 6 entity: 6 ids unique: True


In [7]:
AMOUNT_RANGES = {"card_payment":[5, 250], "transfer_in":[2000, 4000], "transfer_out": [300, 1500], "cash_in": [50, 500], "cash_out": [20, 300]}
TYPE_WEIGHTS = {"card_payment":6, "transfer_in":1, "transfer_out":2, "cash_in": 1, "cash_out": 2}

In [8]:
def make_ordinary_transactions(seed):
    rng = random.Random(seed)
    n = rng.randint(15, 30)
    transactions = []

    types = list(TYPE_WEIGHTS.keys())
    weights = list(TYPE_WEIGHTS.values())

    for _ in range(n):
        picked = rng.choices(types, weights=weights)
        tx_type = picked[0]
        low, high = AMOUNT_RANGES[tx_type]
        amount = round(rng.uniform(low, high), 2)
        days_ago = rng.randint(0, 59)
        tx_date = AS_OF - timedelta(days=days_ago)
        transactions.append({"type": tx_type,
                             "amount": amount,
                             "currency": "EUR",
                             "date": tx_date.isoformat()})
    return transactions

In [9]:
a = make_ordinary_transactions(7)
b = make_ordinary_transactions(7)



print(f"same result: {a == b}")
print(f"first three transactions: {a[:3]}")
print(f"count: {len(a)}")
print(f"min transaction date: {min(list(item['date'] for item in a))}")
print(f"max transaction date: {max(list(item['date'] for item in a))}")

same result: True
first three transactions: [{'type': 'cash_out', 'amount': 130.55, 'currency': 'EUR', 'date': '2026-10-29'}, {'type': 'card_payment', 'amount': 136.29, 'currency': 'EUR', 'date': '2026-10-09'}, {'type': 'transfer_in', 'amount': 3819.41, 'currency': 'EUR', 'date': '2026-10-19'}]
count: 25
min transaction date: 2026-09-10
max transaction date: 2026-10-29


In [10]:
def finalize_transactions(customer_id, transactions):
    transactions.sort(key=lambda tx: tx["date"])
    for i, tx in enumerate(transactions, start=1):
        tx["id"] = f"{customer_id}-T{i:03d}"

    return transactions

In [11]:
SUSPICIOUS_DEPOSITS = {
    # --- structuring: 3+ cash_in, 9,000-9,999.99, span <= 29 days ---
    "C-01": [  # basic case: 3 deposits within one week
        (20, 9450.00, "cash_in"),
        (17, 9720.50, "cash_in"),
        (14, 9380.25, "cash_in"),
    ],
    "C-02": [  # 4 deposits over two weeks
        (45, 9120.00, "cash_in"),
        (40, 9875.40, "cash_in"),
        (36, 9640.10, "cash_in"),
        (31, 9305.75, "cash_in"),
    ],
    "C-07": [  # hard: span is exactly 29 days (edge of the window)
        (50, 9550.00, "cash_in"),
        (35, 9210.30, "cash_in"),
        (21, 9790.60, "cash_in"),
    ],
    "C-09": [  # hard: amount is exactly 9,999.99 (upper edge)
        (12, 9999.99, "cash_in"),
        (9, 9600.00, "cash_in"),
        (5, 9480.20, "cash_in"),
    ],
    "C-12": [  # hard: amount is exactly 9,000.00 (lower edge)
        (28, 9000.00, "cash_in"),
        (24, 9340.80, "cash_in"),
        (20, 9715.45, "cash_in"),
        (16, 9250.00, "cash_in"),
    ],
    "C-18": [  # last deposit is on AS_OF itself (days_ago = 0)
        (8, 9820.00, "cash_in"),
        (4, 9410.65, "cash_in"),
        (0, 9690.30, "cash_in"),
    ],
    # --- decoys: look similar, but are NOT structuring ---
    "C-19": [  # only 2 deposits
        (15, 9700.00, "cash_in"),
        (11, 9850.50, "cash_in"),
    ],
    "C-03": [  # 3 deposits, but span is 30 days (one day too long)
        (55, 9500.00, "cash_in"),
        (40, 9650.25, "cash_in"),
        (25, 9420.80, "cash_in"),
    ],
    "C-11": [  # amounts above 9,999.99 (outside the structuring range)
        (18, 10250.00, "cash_in"),
        (14, 11400.50, "cash_in"),
        (10, 10875.20, "cash_in"),
    ],
    "C-15": [  # transfer_in, not cash_in (rule needs cash)
        (22, 9600.00, "transfer_in"),
        (19, 9450.75, "transfer_in"),
        (15, 9780.30, "transfer_in"),
    ],
}

print(f"suspicious customers: {len(SUSPICIOUS_DEPOSITS)}")

suspicious customers: 10


In [12]:
CUSTOMERS = []
for i in range(1, 21):
    customer_id = f"C-{i:02d}"
    transactions = make_ordinary_transactions(seed=i)
    if customer_id in SUSPICIOUS_DEPOSITS:
        for days_ago, amount, tx_type in SUSPICIOUS_DEPOSITS[customer_id]:
            tx_date = AS_OF - timedelta(days=days_ago)
            transactions.append({"type": tx_type,
                                 "amount": amount,
                                 "currency": "EUR",
                                 "date": tx_date.isoformat()
                                })
    transactions = finalize_transactions(customer_id, transactions)
    CUSTOMERS.append({"id": customer_id, "transactions": transactions})

print(f"customers: {len(CUSTOMERS)}")
assert len(CUSTOMERS) == 20, f"customers: got {len(CUSTOMERS)}"

customer_ids = [c["id"] for c in CUSTOMERS]
assert len(CUSTOMERS) == len(set(customer_ids)), "customer ids are not unique"

print(f"all customers unique: {len(set(cus['id'] for cus in CUSTOMERS)) == len(CUSTOMERS)}")
counts = [len(c["transactions"]) for c in CUSTOMERS]
print(f"all transactions: {sum(counts)}")
assert sum(counts) == 472, f"transactions: got {sum(counts)}"

print(f"minimum number of transactions: {min(counts)}")
print(f"maximum number of transactions: {max(counts)}")
print(f"the first customer id: {CUSTOMERS[0]['id']}")
print(f"the last customer id: {CUSTOMERS[-1]['id']}")

c07 = [c for c in CUSTOMERS if c["id"] == "C-07"][0]
big = [tx for tx in c07["transactions"] if tx["amount"] >= 9000 and tx["type"] == "cash_in"]

print(f"big transactions: {len(big)}")
c05 = [c for c in CUSTOMERS if c["id"] == "C-05"][0]
print(f"C-05 transactions: {len(c05['transactions'])}")

customers: 20
all customers unique: True
all transactions: 472
minimum number of transactions: 16
maximum number of transactions: 34
the first customer id: C-01
the last customer id: C-20
big transactions: 3
C-05 transactions: 23


In [13]:
WORLD = {"as_of": AS_OF.isoformat(), "sanctions": SANCTIONS, "customers": CUSTOMERS}

with open(WORLD_PATH, "w", encoding="utf-8") as f:
    json.dump(WORLD, f, indent=2, ensure_ascii=False, sort_keys=True)
print(f"file saved under {WORLD_PATH}")
print("world file size in bytes:", WORLD_PATH.stat().st_size)

file saved under compliance_eval/data/world.json
world file size in bytes: 84442


## Build the agent's tools

In this section, you build two tools for the agent: `screen_name` checks the payee against the sanctions list, and `get_customer_activity` returns the customer's transaction history.

In [14]:
LEGAL_SUFFIXES = {
    "ltd", "gmbh", "sa",
    "llc", "ag", "bv",
    "nv", "vof", "cv",
    "plc", "llp","corp",
    "corporation", "inc",
    "limited"
}

NORMALIZE_CASES = [
    ("Sablebrook Transit LTD", "sablebrook transit"), # suffix + "sa" inside a word
    ("Zorán El-Märi", "zoran el mari"), # - and diacritics
    ("  Ivo       Nakamberi  ", "ivo nakamberi"), #extra spaces
    ("Cindervale Materials AG", "cindervale materials"), # suffix
    ("Mirela Söderqvist", "mirela soderqvist"), # diacritics
    ("Westfen Logistics SA", "westfen logistics"), #suffix "sa"
    ("Mossfjord Shipping Ltd.", "mossfjord shipping"), #suffix with "."
    ("ESmé KoWaLeNga", "esme kowalenga"), # letters in different register
    ("Veldhollow Marine B.V.", "veldhollow marine"), #suffix
]

SCREEN_FLOOR = 0.70
MAX_MATCHES = 3

In [15]:
def normalize(name):
    name = name.lower()
    decomposed = unicodedata.normalize("NFKD", name)
    name = "".join(ch for ch in decomposed if not unicodedata.combining(ch))
    words = name.split()
    if words:
        words[-1] = words[-1].replace(".","")
    name = " ".join(words)
    name = re.sub(r"[^\w\s]", " ", name)

    words = name.split()

    if words and words[-1] in LEGAL_SUFFIXES:
        words = words[:-1]

    return " ".join(words)

In [16]:
passed = 0
for case, expected in NORMALIZE_CASES:
    got = normalize(case)
    if got == expected:
        passed += 1
    else:
        print(f"FAIL input: {case!r} expected: {expected!r} got: {got!r}")

print(f"passed: {passed} from {len(NORMALIZE_CASES)}")
assert passed == len(NORMALIZE_CASES), f"normalize: {passed} / {len(NORMALIZE_CASES)} cases passed"

passed: 9 from 9


In [17]:
def similarity(a, b):
    normalized_a = normalize(a)
    normalized_b = normalize(b)

    return round(difflib.SequenceMatcher(None, normalized_a, normalized_b).ratio(), 2)

def score_record(query, record):
    candidates = [record["name"]] + record["aliases"]
    best_name = max(candidates, key=lambda c: similarity(query, c))
    score = similarity(query, best_name)

    return best_name, score

In [18]:
san03 = [s for s in SANCTIONS if s["id"] == "SAN-003"][0]
print(f"Mirela Soederqvist in SAN-003: {score_record('Mirela Soederqvist', san03)}")
print(f"Peter Jansen in SAN-003: {score_record('Peter Jansen', san03)}")

san11 = [s for s in SANCTIONS if s["id"] == "SAN-011"][0]
print(f"Esme Kowalenga in SAN-011: {score_record('Esme Kowalenga', san11)}")

Mirela Soederqvist in SAN-003: ('Mirela Soederqvist', 1.0)
Peter Jansen in SAN-003: ('Mirela Söderqvist', 0.28)
Esme Kowalenga in SAN-011: ('Esmé Kowalenga', 1.0)


In [19]:
def screen_name(name):
    result = []
    for sanction in SANCTIONS:
        matched_name, score = score_record(name, sanction)
        if score >= SCREEN_FLOOR:
            result.append({
                "id": sanction["id"],
                "matched_name": matched_name,
                "score": score
            })
    result.sort(key=lambda r: r["score"], reverse=True)

    return result[:MAX_MATCHES]

In [20]:
print(f"A name in the sanctions list: {screen_name('Mirela Soederqvist')}")
print(f"A name not in the sanctions list: {screen_name('Peter Jansen')}")
print(f"Alias match, no false hit on SAN-004: {screen_name('Mossfjord Maritime')}")

A name in the sanctions list: [{'id': 'SAN-003', 'matched_name': 'Mirela Soederqvist', 'score': 1.0}]
A name not in the sanctions list: []
Alias match, no false hit on SAN-004: [{'id': 'SAN-002', 'matched_name': 'Mossfjord Maritime', 'score': 1.0}]


In [21]:
PROBES = [
    ("Aldona Qureshi", "block"), # exact name
    ("Esme Kowalenga", "block"), # without diacritics
    ("Cindervale Materials Corp.", "block"), # with different company
    ("Brackenfyord Analytics", "block"), # one misspell is still block
    ("Esmi Kowalenga", "block"), # one misspell in a short name
    ("Uvo Nakambery", "escalate"), # two misspells
    ("Frank Marine GmbH", "clear"), # clean name
    ("Westfen Logistics Corporation", "block"), # same name with different company
    ("Uvo Nakanbery", "clear") # three misspell
]

In [22]:
def zone(score):
    if score >= BLOCK_THRESHOLD:
        return "block"
    elif score >= ESCALATE_THRESHOLD:
        return "escalate"
    else:
        return "clear"

In [23]:
counter = 0
for probe_name, probe_zone in PROBES:
    result = screen_name(probe_name)
    if result:
        best_score = result[0]["score"]
    else:
        best_score = 0

    zone_result = zone(best_score)
    if zone_result == probe_zone:
        counter += 1

    print(f"{probe_name!r}: expected {probe_zone}, got {zone_result}, score {best_score}")


print(f"probes matched: {counter} from {len(PROBES)}")
assert counter == len(PROBES), f"probes: {counter} / {len(PROBES)} as expected"

'Aldona Qureshi': expected block, got block, score 1.0
'Esme Kowalenga': expected block, got block, score 1.0
'Cindervale Materials Corp.': expected block, got block, score 1.0
'Brackenfyord Analytics': expected block, got block, score 0.95
'Esmi Kowalenga': expected block, got block, score 0.93
'Uvo Nakambery': expected escalate, got escalate, score 0.85
'Frank Marine GmbH': expected clear, got clear, score 0
'Westfen Logistics Corporation': expected block, got block, score 1.0
'Uvo Nakanbery': expected clear, got clear, score 0.77
probes matched: 9 from 9


In [24]:
def get_customer_activity(customer_id):
    with open(WORLD_PATH, encoding="utf-8") as f:
        world = json.load(f)
    matches = [c for c in world["customers"] if c["id"] == customer_id]
    if not matches:
        return { "error": f"unknown customer: {customer_id}"}
    customer = matches[0]
    return {"customer_id": customer_id, "as_of": world["as_of"], "transactions": customer["transactions"]}

In [25]:
c07_activity = get_customer_activity("C-07")
print(f"C-07 transactions: {len(c07_activity['transactions'])}")
print(f"C-07 as_of: {c07_activity['as_of']}")

unknown = get_customer_activity("C-99")
print(f"C-99 result: {unknown}")

C-07 transactions: 28
C-07 as_of: 2026-11-01
C-99 result: {'error': 'unknown customer: C-99'}


In [26]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "screen_name",
            "description": "Check a payee name against the sanctions list. Returns up to 3 matches with a score of 0.70 or higher, each with the sanctions record id, the matched name, and a similarity score (1.0 is an exact match). An empty list means no name scored 0.70 or higher.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Name of the payee (the person or company that receives the payment), for example 'Mossfjord Shipping'.",
                    },
                },
                "required": ["name"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_activity",
            "description": "Get the transactions of a bank customer for the last 60 days. Returns the customer_id, the as_of date (the date of this check), and a list of transactions. If the customer id is unknown, returns an object with an 'error' field.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "Id of the bank customer who sends the payment, for example 'C-07'.",
                    },
                },
                "required": ["customer_id"]
            },
        },
    }
]

## Label the test cases

In this section, you write the policy as code and use it to label 39 test cases with the expected decision: block, escalate, or clear. A case is hard when the payee is a similar but not exact match, or when the customer's deposits are on the edge of the structuring rule or just outside it.

In [27]:
STRUCTURING_MIN_AMOUNT = 9000
STRUCTURING_MAX_AMOUNT = 9999.99
STRUCTURING_WINDOW_DAYS = 29
STRUCTURING_MIN_COUNT = 3

def find_structuring_deposits(transactions):
    deposits = [
        t for t in transactions
        if t["type"] == "cash_in"
        and STRUCTURING_MIN_AMOUNT <= t["amount"] <= STRUCTURING_MAX_AMOUNT
    ]
    deposits = sorted(deposits, key=lambda t: t["date"])

    for i in range(len(deposits) - (STRUCTURING_MIN_COUNT - 1)):
        first = date.fromisoformat(deposits[i]["date"])
        last = date.fromisoformat(deposits[i + STRUCTURING_MIN_COUNT - 1]["date"])
        span_days = (last - first).days

        if span_days <= STRUCTURING_WINDOW_DAYS:
            window_end = first + timedelta(days=STRUCTURING_WINDOW_DAYS)
            return [
                t for t in deposits[i:]
                if date.fromisoformat(t["date"]) <= window_end
            ]

    return []

In [28]:
EXPECTED_STRUCTURING = {
    "C-01", "C-02",
    "C-07", "C-09",
    "C-12", "C-18"
}

mismatches = []
found = []
for customer in CUSTOMERS:
    got = find_structuring_deposits(get_customer_activity(customer["id"])["transactions"])
    expected = customer["id"] in EXPECTED_STRUCTURING
    if bool(got) != expected:
        mismatches.append(customer["id"])
    if got:
        found.append(customer["id"])
print("structuring found:", found)
print("mismatches (expect []):", mismatches)
assert mismatches == [], f"structuring mismatches: {mismatches}"

structuring found: ['C-01', 'C-02', 'C-07', 'C-09', 'C-12', 'C-18']
mismatches (expect []): []


In [29]:
TRACE_DEPOSITS = [
    {"type": "cash_in", "amount": 9500.0, "date": "2026-09-01"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-20"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-25"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-28"},
]
trace_found = find_structuring_deposits(TRACE_DEPOSITS)
trace_dates = [d["date"] for d in trace_found]

print("trace, 4 deposits (expect 2026-10-20, 2026-10-25, 2026-10-28):", trace_dates)
print("trace, 1 deposit (expect []):", find_structuring_deposits(TRACE_DEPOSITS[:1]))
assert trace_dates == ["2026-10-20", "2026-10-25", "2026-10-28"], f"trace: got {trace_dates}"
assert find_structuring_deposits(TRACE_DEPOSITS[:1]) == [], "trace: one deposit must give []"

trace, 4 deposits (expect 2026-10-20, 2026-10-25, 2026-10-28): ['2026-10-20', '2026-10-25', '2026-10-28']
trace, 1 deposit (expect []): []


In [30]:
WINDOW_DEPOSITS = [
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-01"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-05"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-10"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-10-25"},
    {"type": "cash_in", "amount": 9500.0, "date": "2026-11-05"},
]

window_dates = [d["date"] for d in find_structuring_deposits(WINDOW_DEPOSITS)]
expected = ["2026-10-01", "2026-10-05", "2026-10-10", "2026-10-25"]

print("window, 4 inside and 1 after:", window_dates)
assert window_dates == expected, f"window: got {window_dates}"

window, 4 inside and 1 after: ['2026-10-01', '2026-10-05', '2026-10-10', '2026-10-25']


In [31]:
def sanctions_signal(payee):
    matches = screen_name(payee)
    if not matches:
        return "clear", f"no sanctions match for '{payee}'"
    top = matches[0]
    decision = zone(top["score"])
    if decision == "clear":
        evidence = (f"payee '{payee}' is closest to {top['id']} ('{top['matched_name']}') "
                    f"with score {top['score']}, below the escalate threshold {ESCALATE_THRESHOLD}"
                   )
    else:
        evidence = f"payee '{payee}' matches {top['id']} ('{top['matched_name']}') with score {top['score']}"
    return decision, evidence

In [32]:
print("block case:", sanctions_signal("Aldona Qureshi"))
print("escalate case:", sanctions_signal("Uvo Nakambery"))
print("clear case:", sanctions_signal("Frank Marine GmbH"))
print("clear with weak match case:", sanctions_signal("Uvo Nakanbery"))

block case: ('block', "payee 'Aldona Qureshi' matches SAN-001 ('Aldona Qureshi') with score 1.0")
escalate case: ('escalate', "payee 'Uvo Nakambery' matches SAN-007 ('Ivo Nakamberi') with score 0.85")
clear case: ('clear', "no sanctions match for 'Frank Marine GmbH'")
clear with weak match case: ('clear', "payee 'Uvo Nakanbery' is closest to SAN-007 ('Ivo Nakamberi') with score 0.77, below the escalate threshold 0.8")


In [33]:
def structuring_signal(customer_id):
    activity = get_customer_activity(customer_id)

    if "transactions" not in activity:
        raise ValueError(activity["error"])

    deposits = find_structuring_deposits(activity["transactions"])

    if not deposits:
        return "clear", f"no {STRUCTURING_MIN_COUNT}+ cash deposits between {STRUCTURING_MIN_AMOUNT} and {STRUCTURING_MAX_AMOUNT} within {STRUCTURING_WINDOW_DAYS} days for customer {customer_id}"

    deposit_list = ", ".join(
        f"{d['id']} ({d['date']}, {d['amount']})" for d in deposits
    )
    evidence = (
        f"customer {customer_id} made {len(deposits)} cash deposits "
        f"between {STRUCTURING_MIN_AMOUNT} and {STRUCTURING_MAX_AMOUNT} "
        f"within {STRUCTURING_WINDOW_DAYS} days: {deposit_list}"
    )

    return "escalate", evidence

In [34]:
print("structuring C-07:", structuring_signal("C-07"))
print("decoy C-03:", structuring_signal("C-03"))
print("clean C-04:", structuring_signal("C-04"))

try:
    structuring_signal("C-99")
    raised = False
except ValueError as e:
    raised = True
    print("structuring_signal('C-99') raised (expected):", e)
assert raised, "structuring_signal must raise on unknown customer"

structuring C-07: ('escalate', 'customer C-07 made 3 cash deposits between 9000 and 9999.99 within 29 days: C-07-T002 (2026-09-12, 9550.0), C-07-T009 (2026-09-27, 9210.3), C-07-T014 (2026-10-11, 9790.6)')
decoy C-03: ('clear', 'no 3+ cash deposits between 9000 and 9999.99 within 29 days for customer C-03')
clean C-04: ('clear', 'no 3+ cash deposits between 9000 and 9999.99 within 29 days for customer C-04')
structuring_signal('C-99') raised (expected): unknown customer: C-99


In [35]:
DECISION_RANK = {"clear": 1, "escalate": 2, "block": 3}

assert max("clear", "block", key=lambda d: DECISION_RANK[d]) == "block", "block must beat clear"
assert max("escalate", "block", key=lambda d: DECISION_RANK[d]) == "block", "block must beat escalate"
assert max("clear", "escalate", key=lambda d: DECISION_RANK[d]) == "escalate", "escalate must beat clear"
print("decision rank checks passed")

decision rank checks passed


In [36]:
def reference_label(case):
    sanctions_decision, sanctions_evidence = sanctions_signal(case["payee"])
    structuring_decision, structuring_evidence = structuring_signal(case["customer_id"])
    decision = max(sanctions_decision, structuring_decision, key=lambda d: DECISION_RANK[d])
    evidence = f"Sanctions: {sanctions_evidence}. Structuring: {structuring_evidence}"

    return decision, evidence

In [37]:
test_case = {"customer_id": "C-07", "payee": "Aldona Qureshi"}
print("reference label:", reference_label(test_case))

reference label: ('block', "Sanctions: payee 'Aldona Qureshi' matches SAN-001 ('Aldona Qureshi') with score 1.0. Structuring: customer C-07 made 3 cash deposits between 9000 and 9999.99 within 29 days: C-07-T002 (2026-09-12, 9550.0), C-07-T009 (2026-09-27, 9210.3), C-07-T014 (2026-10-11, 9790.6)")


In [38]:
LABEL_CHECKS = [
    {"customer_id": "C-07", "payee": "Aldona Qureshi", "expected": "block"},
    {"customer_id": "C-01", "payee": "Uvo Nakanbery", "expected": "escalate"},
    {"customer_id": "C-03", "payee": "Uvo Nakambery", "expected": "escalate"},
    {"customer_id": "C-01", "payee": "Uvo Nakambery", "expected": "escalate"},
    {"customer_id": "C-03", "payee": "Frank Marine GmbH", "expected": "clear"},
    {"customer_id": "C-04", "payee": "Uvo Nakanbery", "expected": "clear"}
]

In [39]:
label_mismatches = []
for check in LABEL_CHECKS:
    case = {"customer_id": check["customer_id"], "payee": check["payee"]}
    decision, evidence = reference_label(case)
    print(f"{check['customer_id']} -> {check['payee']}: {decision} (expected {check['expected']})")
    if decision != check["expected"]:
        label_mismatches.append(check)
print("label mismatches:", label_mismatches)
assert label_mismatches == [], f"label mismatches: {label_mismatches}"

C-07 -> Aldona Qureshi: block (expected block)
C-01 -> Uvo Nakanbery: escalate (expected escalate)
C-03 -> Uvo Nakambery: escalate (expected escalate)
C-01 -> Uvo Nakambery: escalate (expected escalate)
C-03 -> Frank Marine GmbH: clear (expected clear)
C-04 -> Uvo Nakanbery: clear (expected clear)
label mismatches: []


In [40]:
CANDIDATE_NAMES = [
    "Tenzin Volimaa",
    "Tenzin Vlaimaa",
    "Pencin Pälimaa",
    "Sablebrook Bransit LTD",
    "Cablebrook Transet LTD",
    "Vildholloff Marene GmbH",
    "Frank Marine GmbH",
    "Ilse Brandvold",
    "Teodor Marcu",
    "Nadia Okonkwo-Reis",
    "Pieter Halvorsen",
    "Kestrel Grain B.V.",
    "Duneholm Textiles GmbH",
    "Oakridge Paper LTD",
    "Valtor Ceramics AG",
    "Aldona Kyreshi",
    "Mohhfjord Maritime",
    "Yesmé Kawalenga",
    "Cindervale Industrial",
    "Ivo Nakamberi",
    "Westfen Logistics SA",
    "Nirela Söderqvist",
    "Zorán El-Märi",
    "Sableboook Drancit LTD",
    "Brackenfgord Anonytics LLC",
    "Vemdhollow Meryne GmbH",
    "Vectgen Mogistics SA",
    "Brackenfjord Analytics",
    "Brackenfjord Data",
    "Qendhollow Meryne GmbH",
    "Vestgen Kogistics SA",
    "Vildholloff Murene GmbH",
    "Antona Poreshi",
    "Serela Sudetqvist",
    "Cintertole Maceriats AG",
    "Lorán Ol-Meri",
    "Pratkemfjort Anamytics LLC",
    "Brackendjord Analytics LLC",
    "Migela Söderqvist",
    "Uvo Nakambery",
    "Uvo Nakanbery",
    "Aldona Qureshi"
]

decisions = []
for name in CANDIDATE_NAMES:
    decision, evidence = sanctions_signal(name)
    decisions.append(decision)
print("candidate names:", len(CANDIDATE_NAMES), "by decision:", Counter(decisions))

candidate names: 42 by decision: Counter({'clear': 18, 'block': 14, 'escalate': 10})


In [41]:
BOUNDARY_CUSTOMERS = {
    "C-19", "C-03", "C-11", "C-15",
    "C-07", "C-09", "C-12", "C-18"
}

def is_hard(customer_id, payee):
    matches = screen_name(payee)
    if matches:
        payee_on_boundary = matches[0]["score"] < 1.0
    else:
        payee_on_boundary = False
    customer_boundary = customer_id in BOUNDARY_CUSTOMERS
    return payee_on_boundary or customer_boundary

In [42]:
print("C-04 + Frank Marine (expect False):", is_hard("C-04", "Frank Marine GmbH"))
print("C-03 + Frank Marine (expect True):", is_hard("C-03", "Frank Marine GmbH"))
print("C-04 + Tenzin Volimaa (expect True):", is_hard("C-04", "Tenzin Volimaa"))
print("C-04 + Esmé Kowalenga (expect False):", is_hard("C-04", "Esmé Kowalenga"))

assert is_hard("C-03", "Frank Marine GmbH"), "decoy customer must be hard"
assert not is_hard("C-04", "Frank Marine GmbH"), "clean + clean must be easy"
assert is_hard("C-04", "Tenzin Volimaa"), "typo payee must be hard"
assert not is_hard("C-04", "Esmé Kowalenga"), "exact match must be easy"

C-04 + Frank Marine (expect False): False
C-03 + Frank Marine (expect True): True
C-04 + Tenzin Volimaa (expect True): True
C-04 + Esmé Kowalenga (expect False): False


In [43]:
CASE_PLAN = [
    # block, exact match, clean customer (6) - easy
    {"customer_id": "C-04", "payee": "Aldona Qureshi", "amount": 2500, "intended": "block"},
    {"customer_id": "C-05", "payee": "Cindervale Industrial", "amount": 1250, "intended": "block"},
    {"customer_id": "C-06", "payee": "Ivo Nakamberi", "amount": 2600, "intended": "block"},
    {"customer_id": "C-13", "payee": "Westfen Logistics SA", "amount": 600, "intended": "block"},
    {"customer_id": "C-17", "payee": "Zorán El-Märi", "amount": 6100, "intended": "block"},
    {"customer_id": "C-20", "payee": "Brackenfjord Analytics", "amount": 890, "intended": "block"},
    # block, boundary payee + boundary structuring customer (4) - both signals fire
    {"customer_id": "C-07", "payee": "Tenzin Volimaa", "amount": 1890, "intended": "block"},
    {"customer_id": "C-09", "payee": "Tenzin Vlaimaa", "amount": 2300, "intended": "block"},
    {"customer_id": "C-12", "payee": "Sablebrook Bransit LTD", "amount": 200, "intended": "block"},
    {"customer_id": "C-18", "payee": "Aldona Kyreshi", "amount": 4399, "intended": "block"},
    # block, boundary payee + clean customer (3)
    {"customer_id": "C-20", "payee": "Nirela Söderqvist", "amount": 1399, "intended": "block"},
    {"customer_id": "C-16", "payee": "Brackendjord Analytics LLC", "amount": 3250, "intended": "block"},
    {"customer_id": "C-14", "payee": "Migela Söderqvist", "amount": 2250, "intended": "block"},
    # escalate, sanctions only: escalate-zone payee + clean or decoy customer (4)
    {"customer_id": "C-10", "payee": "Cablebrook Transet LTD", "amount": 2750, "intended": "escalate"},
    {"customer_id": "C-03", "payee": "Vildholloff Marene GmbH", "amount": 3750, "intended": "escalate"},
    {"customer_id": "C-08", "payee": "Mohhfjord Maritime", "amount": 4750, "intended": "escalate"},
    {"customer_id": "C-19", "payee": "Yesmé Kawalenga", "amount": 750, "intended": "escalate"},
    # escalate, structuring only: clean payee + structuring customer (4) - easy
    {"customer_id": "C-01", "payee": "Frank Marine GmbH", "amount": 2755, "intended": "escalate"},
    {"customer_id": "C-02", "payee": "Ilse Brandvold", "amount": 3000, "intended": "escalate"},
    {"customer_id": "C-01", "payee": "Teodor Marcu", "amount": 300, "intended": "escalate"},
    {"customer_id": "C-02", "payee": "Nadia Okonkwo-Reis", "amount": 1890, "intended": "escalate"},
    # escalate, both signals: escalate-zone payee + structuring customer (5)
    {"customer_id": "C-01", "payee": "Uvo Nakambery", "amount": 4900, "intended": "escalate"},
    {"customer_id": "C-02", "payee": "Lorán Ol-Meri", "amount": 4123, "intended": "escalate"},
    {"customer_id": "C-07", "payee": "Vestgen Kogistics SA", "amount": 3789, "intended": "escalate"},
    {"customer_id": "C-09", "payee": "Vemdhollow Meryne GmbH", "amount": 1256, "intended": "escalate"},
    {"customer_id": "C-12", "payee": "Brackenfgord Anonytics LLC", "amount": 1050, "intended": "escalate"},
    # clear, near-miss payee (0.70-0.79) + decoy customer (6)
    {"customer_id": "C-03", "payee": "Pencin Pälimaa", "amount": 900, "intended": "clear"},
    {"customer_id": "C-19", "payee": "Vectgen Mogistics SA", "amount": 1900, "intended": "clear"},
    {"customer_id": "C-11", "payee": "Qendhollow Meryne GmbH", "amount": 2599, "intended": "clear"},
    {"customer_id": "C-11", "payee": "Vildholloff Murene GmbH", "amount": 120, "intended": "clear"},
    {"customer_id": "C-15", "payee": "Antona Poreshi", "amount": 340, "intended": "clear"},
    {"customer_id": "C-15", "payee": "Serela Sudetqvist", "amount": 8900, "intended": "clear"},
    # clear, clean payee + clean customer (2 of 5) - easy
    {"customer_id": "C-10", "payee": "Pieter Halvorsen", "amount": 3900, "intended": "clear"},
    {"customer_id": "C-08", "payee": "Kestrel Grain B.V.", "amount": 3000, "intended": "clear"},
    # clear, near-miss payee (0.70-0.79) + clean customer (2)
    {"customer_id": "C-17", "payee": "Cintertole Maceriats AG", "amount": 4500, "intended": "clear"},
    {"customer_id": "C-16", "payee": "Pratkemfjort Anamytics LLC", "amount": 450, "intended": "clear"},
    # clear, clean payee + clean customer (3 of 5) - easy
    {"customer_id": "C-14", "payee": "Duneholm Textiles GmbH", "amount": 3270, "intended": "clear"},
    {"customer_id": "C-13", "payee": "Oakridge Paper LTD", "amount": 2800, "intended": "clear"},
    {"customer_id": "C-05", "payee": "Valtor Ceramics AG", "amount": 1800, "intended": "clear"},
]

In [44]:
CASES = []
for i, plan in enumerate(CASE_PLAN, start=1):
    decision, evidence = reference_label(plan)
    case = {
        "id": f"case-{i:02d}",
        "request": f"Customer {plan['customer_id']} wants to send €{plan['amount']:,} to {plan['payee']}.",
        "customer_id": plan["customer_id"],
        "payee": plan["payee"],
        "amount": plan["amount"],
        "expected_decision": decision,
        "evidence": evidence,
        "difficulty": "hard" if is_hard(plan["customer_id"], plan["payee"]) else "easy",
    }
    CASES.append(case)
print("cases built:", len(CASES))

cases built: 39


In [45]:
intent_mismatches = []
for plan, case in zip(CASE_PLAN, CASES):
    if plan["intended"] != case["expected_decision"]:
        intent_mismatches.append((case["id"], plan["payee"], plan["intended"], case["expected_decision"]))
print("intent mismatches:", intent_mismatches)
assert intent_mismatches == [], f"intent mismatches: {intent_mismatches}"

label_split = Counter(c["expected_decision"] for c in CASES)
difficulty_split = Counter(c["difficulty"] for c in CASES)
print("label split:", label_split)
print("difficulty_split:", difficulty_split)
assert label_split == {"block": 13, "escalate": 13, "clear": 13}, f"label_split: {label_split}"
assert difficulty_split == {"hard": 24, "easy": 15}, f"difficulty_split: {difficulty_split}"

intent mismatches: []
label split: Counter({'block': 13, 'escalate': 13, 'clear': 13})
difficulty_split: Counter({'hard': 24, 'easy': 15})


In [46]:
hard = [c for c in CASES if c["difficulty"] == "hard"]
print("hard cases:", len(hard), "(first 3 shown)")
for c in hard[:3]:
    print(f"{c['id']} | {c['expected_decision']} | {c['request']}")
    print(f"    evidence: {c['evidence']}")

hard cases: 24 (first 3 shown)
case-07 | block | Customer C-07 wants to send €1,890 to Tenzin Volimaa.
    evidence: Sanctions: payee 'Tenzin Volimaa' matches SAN-005 ('Tenzin Välimaa') with score 0.93. Structuring: customer C-07 made 3 cash deposits between 9000 and 9999.99 within 29 days: C-07-T002 (2026-09-12, 9550.0), C-07-T009 (2026-09-27, 9210.3), C-07-T014 (2026-10-11, 9790.6)
case-08 | block | Customer C-09 wants to send €2,300 to Tenzin Vlaimaa.
    evidence: Sanctions: payee 'Tenzin Vlaimaa' matches SAN-005 ('Tenzin Välimaa') with score 0.93. Structuring: customer C-09 made 3 cash deposits between 9000 and 9999.99 within 29 days: C-09-T019 (2026-10-20, 9999.99), C-09-T023 (2026-10-23, 9600.0), C-09-T028 (2026-10-27, 9480.2)
case-09 | block | Customer C-12 wants to send €200 to Sablebrook Bransit LTD.
    evidence: Sanctions: payee 'Sablebrook Bransit LTD' matches SAN-012 ('Sablebrook Transit LTD') with score 0.94. Structuring: customer C-12 made 4 cash deposits between 9000 a

In [47]:
CASES_PATH = DATA_DIR / "cases.jsonl"

with open(CASES_PATH, "w", encoding="utf-8") as f:
    for c in CASES:
        f.write(json.dumps(c, ensure_ascii=False, sort_keys=True) + "\n")
print("cases written:", CASES_PATH, "| size in bytes:", CASES_PATH.stat().st_size)

loaded = [json.loads(line) for line in CASES_PATH.read_text(encoding="utf-8").splitlines()]
print("cases read back:", len(loaded))
assert loaded == CASES, "cases.jsonl does not match CASES after reading back"

cases written: compliance_eval/data/cases.jsonl | size in bytes: 17585
cases read back: 39


## Build the scorer

In this section, you build the pass rate. A case passes when the agent's decision matches the expected decision, checked by code, and an LLM judge says the reason is grounded. The rubric, the judge, and the scoring code are written to files, so oathgate can freeze them later.

In [48]:
%%writefile compliance_eval/judge_rubric.md
# Judge rubric

You check one answer from a compliance agent.
You get: the request, the evidence, the agent's decision, and the agent's reason.
The decision is already correct. You check only the reason.
For rules 2 and 3, look only at the agent's reason, not at the evidence.

The reason is grounded only if all three rules hold:
1. Every fact in the reason matches the evidence. Nothing is invented.
2. The reason names the result of both checks, the sanctions check of the payee and the structuring check of the customer, even a clean one.
3. Each result includes the specific details the evidence gives for it: numbers, ranges, and thresholds.

Check each rule separately. Answer with JSON only:
{"rule_1": true or false, "rule_2": true or false, "rule_3": true or false, "grounded": true only if all three rules are true, "why": "one short sentence"}

Overwriting compliance_eval/judge_rubric.md


In [49]:
%%writefile compliance_eval/judge.py
import hashlib
import json
from pathlib import Path

HERE = Path(__file__).parent
RUBRIC_PATH = HERE / "judge_rubric.md"
CACHE_DIR = HERE / "judge_cache"
JUDGE_MODEL = "mistral-small-2603"
JUDGE_SEED = 13


def load_rubric():
    return RUBRIC_PATH.read_text(encoding="utf-8")


def cache_key(model, rubric, case, decision, reason):
    parts = [model, rubric, case["request"], case["evidence"], decision, reason, str(JUDGE_SEED)]
    text = "\n---\n".join(parts)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def rule_names(rubric):
    return [
        f"rule_{line.split('.')[0]}"
        for line in rubric.splitlines()
        if line[:1].isdigit()
    ]


def judge_messages(rubric, case, decision, reason):
    user_text = (
        f"Request: {case['request']}\n"
        f"Evidence: {case['evidence']}\n"
        f"Agent decision: {decision}\n"
        f"Agent reason: {reason}"
    )
    return [
        {"role": "system", "content": rubric},
        {"role": "user", "content": user_text},
    ]


def parse_judge_reply(text, expected_rules):
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        raise ValueError(f"judge reply is not JSON: {text[:80]}")
    if not isinstance(data, dict):
        raise ValueError(f"judge reply is not a JSON object: {text[:80]}")
    got_rules = sorted(key for key in data if key.startswith("rule_"))
    if got_rules != sorted(expected_rules):
        raise ValueError(f"judge reply has rules {got_rules}, expected {expected_rules}: {text[:80]}")
    if not all(isinstance(data[rule], bool) for rule in expected_rules):
        raise ValueError(f"rule fields must be true or false: {text[:80]}")
    data["grounded"] = all(data[rule] for rule in expected_rules)
    return data


def judge(client, case, decision, reason, cache_dir=CACHE_DIR):
    rubric = load_rubric()
    key = cache_key(JUDGE_MODEL, rubric, case, decision, reason)
    path = cache_dir / f"{key}.json"

    if path.exists():
        text = path.read_text(encoding="utf-8")
        return parse_judge_reply(text, rule_names(rubric))

    response = client.chat.complete(
        model=JUDGE_MODEL,
        messages=judge_messages(rubric, case, decision, reason),
        temperature=0,
        random_seed=JUDGE_SEED,
        response_format={"type": "json_object"},
    )
    text = response.choices[0].message.content
    data = parse_judge_reply(text, rule_names(rubric))

    cache_dir.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data), encoding="utf-8")
    return data

Overwriting compliance_eval/judge.py


In [50]:
%%writefile compliance_eval/pass_rate.py
from compliance_eval import judge


def case_passes(client, case, output, cache_dir=judge.CACHE_DIR):
    decision = output.get("decision")
    if decision != case["expected_decision"]:
        return False

    reason = output.get("reason")
    if not isinstance(reason, str):
        return False

    result = judge.judge(client, case, decision, reason, cache_dir=cache_dir)
    return result["grounded"]


def count_passes(client, cases, outputs, cache_dir=judge.CACHE_DIR):
    n = {"total": 0, "easy": 0, "hard": 0}
    passed = {"total": 0, "easy": 0, "hard": 0}

    for case in cases:
        output = outputs.get(case["id"], {})
        ok = case_passes(client, case, output, cache_dir=cache_dir)
        for group in ["total", case["difficulty"]]:
            n[group] += 1
            if ok:
                passed[group] += 1

    return n, passed


def pass_rate(client, cases, outputs, cache_dir=judge.CACHE_DIR):
    n, passed = count_passes(client, cases, outputs, cache_dir=cache_dir)

    result = {}
    for group in ["total", "easy", "hard"]:
        if n[group] == 0:
            rate = None
        else:
            rate = round(passed[group] / n[group], 2)
        result[group] = {"passed": passed[group], "n": n[group], "rate": rate}
    return result

Overwriting compliance_eval/pass_rate.py


In [51]:
from compliance_eval import judge as judge_module
from compliance_eval import pass_rate as pass_rate_module


def load_cases():
    lines = CASES_PATH.read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines]


def evaluate(client, outputs, cache_dir=None):
    importlib.reload(judge_module)          # judge first:
    importlib.reload(pass_rate_module)      # pass_rate.py imports judge, so it must see the fresh one
    cases = load_cases()                    # from disk, not the CASES list in memory
    if cache_dir is None:
        cache_dir = judge_module.CACHE_DIR
    return pass_rate_module.pass_rate(client, cases, outputs, cache_dir=cache_dir)

In [52]:
class FakeClient:
    def __init__(self, reply):
        self.reply = reply
        self.calls = 0
        self.chat = self

    def complete(self, **kwargs):
        self.calls += 1
        message = SimpleNamespace(content=self.reply)
        choice = SimpleNamespace(message=message)
        return SimpleNamespace(choices=[choice])

JUDGE_PASS = '{"rule_1": true, "rule_2": true, "rule_3": true, "why": "ok"}'
JUDGE_FAIL = '{"rule_1": true, "rule_2": true, "rule_3": false, "why": "no numbers"}'

In [53]:
importlib.reload(judge_module)  # use the file written above
case = CASES[0]
TEST_CACHE = Path("compliance_eval/judge_cache_test")
shutil.rmtree(TEST_CACHE, ignore_errors=True)

# 1. good reply: goes to the client and comes back
fake = FakeClient(JUDGE_PASS)
r1 = judge_module.judge(fake, case, "block", "reason A", cache_dir=TEST_CACHE)
assert r1["grounded"] is True, "fake reply must come back"
assert fake.calls == 1, "first call must reach the client"

# 2. same inputs again: must come from cache, client not called
r2 = judge_module.judge(fake, case, "block", "reason A", cache_dir=TEST_CACHE)
assert fake.calls == 1, "same inputs must come from cache"

# 3. broken reply: must raise and must not be cached
broken = FakeClient("not json")
raised = False
try:
    judge_module.judge(broken, case, "block", "reason B", cache_dir=TEST_CACHE)
except ValueError:
    raised = True
assert raised, "broken reply must raise"

files = list(TEST_CACHE.glob("*.json"))
assert len(files) == 1, "broken reply must not be cached"

# 4. broken cache file: must raise, client not called
key = judge_module.cache_key(judge_module.JUDGE_MODEL, judge_module.load_rubric(), case, "block", "reason C")
(TEST_CACHE / f"{key}.json").write_text('{"grounded": true}', encoding="utf-8")
fake_cache = FakeClient(JUDGE_PASS)
raised = False
try:
    judge_module.judge(fake_cache, case, "block", "reason C", cache_dir=TEST_CACHE)
except ValueError:
    raised = True
assert raised, "broken cache file must raise"
assert fake_cache.calls == 0, "broken cache file must not reach the client"

shutil.rmtree(TEST_CACHE)   # remove test files
print("judge.py with fake client: 4 checks passed")

judge.py with fake client: 4 checks passed


In [54]:
importlib.reload(pass_rate_module)  # use the file written above
case = CASES[0]
TEST_CACHE = Path("compliance_eval/judge_cache_test")
shutil.rmtree(TEST_CACHE, ignore_errors=True)

TESTS = [
    ({"decision": "block", "reason": "reason A"}, JUDGE_PASS, True, 1),
    ({"decision": "block", "reason": "reason B"}, JUDGE_FAIL, False, 1),
    ({"decision": "clear", "reason": "reason C"}, JUDGE_PASS, False, 0),
    ({"reason": "reason D"}, JUDGE_PASS, False, 0),
    ({"decision": "block", "reason": None}, JUDGE_PASS, False, 0),
    ({"decision": "block", "reason": 123}, JUDGE_PASS, False, 0),
    ({"decision": "block", "reason": ["a", "b"]}, JUDGE_PASS, False, 0),
    ({"decision": "block"}, JUDGE_PASS, False, 0),
]

for output, reply, expected, expected_calls in TESTS:
    fake = FakeClient(reply)
    got = pass_rate_module.case_passes(fake, case, output, cache_dir=TEST_CACHE)
    assert got == expected, f"{output}: expected {expected}, got {got}"
    assert fake.calls == expected_calls, f"{output}: expected {expected_calls} calls, got {fake.calls}"

shutil.rmtree(TEST_CACHE)
print(f"pass_rate.py case_passes: {len(TESTS)} checks passed")

pass_rate.py case_passes: 8 checks passed


In [55]:
importlib.reload(pass_rate_module)  # use the file written above
easy_cases = [c for c in CASES if c["difficulty"] == "easy"]
hard_cases = [c for c in CASES if c["difficulty"] == "hard"]

easy_case = easy_cases[0]
hard_ok = hard_cases[0]
hard_bad = hard_cases[1]
small_set = [easy_case, hard_ok, hard_bad]

outputs = {
    easy_case["id"]: {"decision": easy_case["expected_decision"], "reason": "r1"},
    hard_ok["id"]: {"decision": hard_ok["expected_decision"], "reason": "r2"},
    # hard_bad is missing on purpose: the agent gave no answer
}

expected = {
    "total": {"passed": 2, "n": 3, "rate": 0.67},
    "easy": {"passed": 1, "n": 1, "rate": 1.0 },
    "hard": {"passed": 1, "n": 2, "rate": 0.5},
}

TEST_CACHE = Path("compliance_eval/judge_cache_test")
shutil.rmtree(TEST_CACHE, ignore_errors=True)

fake = FakeClient(JUDGE_PASS)
got = pass_rate_module.pass_rate(fake, small_set, outputs, cache_dir=TEST_CACHE)
assert got == expected, f"pass_rate: expected {expected}, got {got}"

shutil.rmtree(TEST_CACHE)
print("pass_rate.py pass_rate on 3 cases:", got)

pass_rate.py pass_rate on 3 cases: {'total': {'passed': 2, 'n': 3, 'rate': 0.67}, 'easy': {'passed': 1, 'n': 1, 'rate': 1.0}, 'hard': {'passed': 1, 'n': 2, 'rate': 0.5}}


In [56]:
importlib.reload(pass_rate_module)  # use the file written above
TEST_CACHE = Path("compliance_eval/judge_cache_test")
shutil.rmtree(TEST_CACHE, ignore_errors=True)

fake = FakeClient(JUDGE_PASS)
got = pass_rate_module.pass_rate(fake, [easy_case], outputs, cache_dir=TEST_CACHE)
assert got["hard"] == {"passed": 0, "n": 0, "rate": None}, f"empty group: got {got['hard']}"

shutil.rmtree(TEST_CACHE)
print("pass_rate.py pass_rate with no hard cases:", got["hard"])

pass_rate.py pass_rate with no hard cases: {'passed': 0, 'n': 0, 'rate': None}


In [57]:
importlib.reload(judge_module)  # use the file written above
rubric_text = judge_module.load_rubric()
model = judge_module.JUDGE_MODEL
case = CASES[0]

k1 = judge_module.cache_key(model, rubric_text, case, "block", "some reason")
k2 = judge_module.cache_key(model, rubric_text, case, "block", "some reason")
k3 = judge_module.cache_key(model, rubric_text + " changed", case, "block", "some reason")

original_seed = judge_module.JUDGE_SEED
try:
    judge_module.JUDGE_SEED = original_seed + 1
    k4 = judge_module.cache_key(model, rubric_text, case, "block", "some reason")
finally:
    judge_module.JUDGE_SEED = original_seed

assert k1 == k2, "same inputs must give the same key"
assert k1 != k3, "a changed rubric must give a different key"
assert k1 != k4, "a changed judge seed must give a different key"
print(f"judge.py cache key example: {k1[:12]}")

judge.py cache key example: e12ec2d4b5a7


In [58]:
# judge_messages from judge.py
msgs = judge_module.judge_messages(rubric_text, CASES[0], "block", "some reason")
assert isinstance(msgs[1]["content"], str), "user content must be one string"
print(f"judge.py user_text:\n{msgs[1]['content']}")

# rule_names from judge.py
rules = judge_module.rule_names(rubric_text)
assert rules == ["rule_1", "rule_2", "rule_3"], f"rule names: {rules}"
print("judge.py rule_names:", rules)

# parse_judge_reply from judge.py
assert judge_module.parse_judge_reply(JUDGE_PASS, rules)["grounded"] is True, "all rules true must pass"
assert judge_module.parse_judge_reply(JUDGE_FAIL, rules)["grounded"] is False, "one false rule must fail"
override = '{"rule_1": true, "rule_2": true, "rule_3": true, "grounded": false}'
assert judge_module.parse_judge_reply(override, rules)["grounded"] is True, "code must compute grounded, not the judge"

bad_replies = [
    '{"rule_1": true, "rule_2": true}',  # rule_3 is missing
    '{"rule_1": true, "rule_2": true, "rule_3": "yes"}',  # rule_3 is not a bool
    '[true, true, true]',  # valid JSON, but not an object
    'hello there',  # not JSON
]

for reply in bad_replies:
    raised = False
    try:
        judge_module.parse_judge_reply(reply, rules)
    except ValueError:
        raised = True
    assert raised, f"must raise for: {reply}"
print(f"judge.py parse_judge_reply: 3 good, {len(bad_replies)} bad replies checked")

judge.py user_text:
Request: Customer C-04 wants to send €2,500 to Aldona Qureshi.
Evidence: Sanctions: payee 'Aldona Qureshi' matches SAN-001 ('Aldona Qureshi') with score 1.0. Structuring: no 3+ cash deposits between 9000 and 9999.99 within 29 days for customer C-04
Agent decision: block
Agent reason: some reason
judge.py rule_names: ['rule_1', 'rule_2', 'rule_3']
judge.py parse_judge_reply: 3 good, 4 bad replies checked


In [59]:
TEST_CACHE = Path("compliance_eval/judge_cache_test")
shutil.rmtree(TEST_CACHE, ignore_errors=True)

# 1. perfect agent: every decision is right
perfect = {c["id"]: {"decision": c["expected_decision"], "reason": "r"} for c in CASES}
fake = FakeClient(JUDGE_PASS)
got = evaluate(fake, perfect, cache_dir=TEST_CACHE)
assert got["total"] == {"passed": 39, "n": 39, "rate": 1.0}, f"perfect total: {got['total']}"
assert got["easy"]["n"] == 15 and got["hard"]["n"] == 24, f"split: {got}"
assert fake.calls == 39, f"perfect: judge calls {fake.calls}"

# 2. silent agent: no answers at all
fake = FakeClient(JUDGE_PASS)
got = evaluate(fake, {}, cache_dir=TEST_CACHE)
assert got["total"] == {"passed": 0, "n": 39, "rate": 0.0}, f"silent total: {got['total']}"
assert fake.calls == 0, f"silent judge calls {fake.calls}"

shutil.rmtree(TEST_CACHE)
print("evaluate from files: 2 runs checked")

evaluate from files: 2 runs checked


In [60]:
judge_case = [
    c for c in CASES
    if c["customer_id"] == "C-01" and c["expected_decision"] == "escalate" and c["difficulty"] == "easy"][0]
print("judge check case:", judge_case["id"], judge_case["request"])
print("judge check evidence:", judge_case["evidence"])

judge check case: case-18 Customer C-01 wants to send €2,755 to Frank Marine GmbH.
judge check evidence: Sanctions: no sanctions match for 'Frank Marine GmbH'. Structuring: customer C-01 made 3 cash deposits between 9000 and 9999.99 within 29 days: C-01-T014 (2026-10-12, 9450.0), C-01-T015 (2026-10-15, 9720.5), C-01-T017 (2026-10-18, 9380.25)


In [61]:
JUDGE_CHECK_A = (
    "Customer C-01 made 3 cash deposits between 9000 and 9999.99 within 29 days: "
    "9450.0 on 2026-10-12, 9720.5 on 2026-10-15, and 9380.25 on 2026-10-18. "
    "The payee 'Frank Marine GmbH' has no sanctions match."
)
JUDGE_CHECKS = [
    ("A, good", JUDGE_CHECK_A, True),
    ("B, vague", "Customer C-01 made 3 cash deposits just under 10,000 within a month. The payee 'Frank Marine GmbH' has no sanctions match.", False),
    ("C, no facts", "Suspicious activity, needs manual review.", False),
    ("D, invented count", JUDGE_CHECK_A.replace("made 3", "made 5"), False),
    ("E, no structuring", "The payee 'Frank Marine GmbH' has no sanctions match.", False),
    ("F, no sanctions", JUDGE_CHECK_A.replace(" The payee 'Frank Marine GmbH' has no sanctions match.", ""), False),
]

assert JUDGE_CHECKS[3][1] != JUDGE_CHECK_A, "D: replace did nothing"
assert JUDGE_CHECKS[5][1] != JUDGE_CHECK_A, "F: replace did nothing"

JUDGE_CHECK_CACHE = Path("compliance_eval/judge_cache_check")
shutil.rmtree(JUDGE_CHECK_CACHE, ignore_errors=True)

agree = 0
for variant, reason, expected in JUDGE_CHECKS:
    verdict = judge_module.judge(client, judge_case, "escalate", reason, cache_dir=JUDGE_CHECK_CACHE)
    if verdict["grounded"] == expected:
        agree += 1
    print(f"judge check {variant}: expected {expected}, judge {verdict['grounded']}, rules {verdict.get('rule_1')}/{verdict.get('rule_2')}/{verdict.get('rule_3')}, why: {verdict.get('why')}")

shutil.rmtree(JUDGE_CHECK_CACHE, ignore_errors=True)
print("judge agrees with my labels:", agree, "of", len(JUDGE_CHECKS))

judge check A, good: expected True, judge True, rules True/True/True, why: Reason matches evidence and includes all required details for both checks.
judge check B, vague: expected False, judge True, rules True/True/True, why: Reason matches evidence and includes all required details.
judge check C, no facts: expected False, judge False, rules True/False/False, why: Reason does not name the results of the sanctions and structuring checks.
judge check D, invented count: expected False, judge False, rules False/True/False, why: Agent incorrectly states 5 deposits instead of 3 and omits the structuring check result for the payee.
judge check E, no structuring: expected False, judge False, rules True/False/False, why: Agent's reason does not mention the structuring check result for customer C-01
judge check F, no sanctions: expected False, judge True, rules True/True/True, why: Reason matches evidence exactly and includes all required details.
judge agrees with my labels: 4 of 6


In [62]:
held_out_case = [
    c for c in CASES
    if c["expected_decision"] == "escalate" and "no 3+ cash deposits" in c["evidence"]][0]
print("held-out case:", held_out_case["id"], held_out_case["request"])
print("held-out evidence:", held_out_case["evidence"])

held-out case: case-14 Customer C-10 wants to send €2,750 to Cablebrook Transet LTD.
held-out evidence: Sanctions: payee 'Cablebrook Transet LTD' matches SAN-012 ('Sablebrook Transit LTD') with score 0.89. Structuring: no 3+ cash deposits between 9000 and 9999.99 within 29 days for customer C-10


In [63]:
HELD_OUT_SANCTIONS = "The payee 'Cablebrook Transet LTD' matches SAN-012 ('Sablebrook Transit LTD') with score 0.89."
HELD_OUT_STRUCTURING = "Customer C-10 has no 3+ cash deposits between 9000 and 9999.99 within 29 days."
HELD_OUT_A = HELD_OUT_SANCTIONS + " " + HELD_OUT_STRUCTURING
HELD_OUT_CHECKS = [
    ("A, good", HELD_OUT_A, True),
    ("B, vague", "The payee name looks similar to a sanctioned company. " + HELD_OUT_STRUCTURING, False),
    ("C, no facts", "Possible risk, needs manual review.", False),
    ("D, invented score", HELD_OUT_A.replace("score 0.89","score 0.95"), False),
    ("E, no sanctions", HELD_OUT_STRUCTURING, False),
    ("F, no structuring", HELD_OUT_SANCTIONS, False),
]
assert HELD_OUT_CHECKS[3][1] != HELD_OUT_A, "D: replace did nothing"

shutil.rmtree(JUDGE_CHECK_CACHE, ignore_errors=True)

agree = 0
for variant, reason, expected in HELD_OUT_CHECKS:
    verdict = judge_module.judge(client, held_out_case, "escalate", reason, cache_dir=JUDGE_CHECK_CACHE)
    if verdict["grounded"] == expected:
        agree += 1
    print(f"held-out {variant}: expected {expected}, judge {verdict['grounded']}, rules {verdict.get('rule_1')}/{verdict.get('rule_2')}/{verdict.get('rule_3')}, why: {verdict.get('why')}")
shutil.rmtree(JUDGE_CHECK_CACHE, ignore_errors=True)
print("held-out: judge agrees with my labels:", agree, "of", len(HELD_OUT_CHECKS))

held-out A, good: expected True, judge True, rules True/True/True, why: All facts match evidence, both checks named with details included.
held-out B, vague: expected False, judge False, rules True/True/False, why: Sanctions result lacks the specific score (0.89) in the reason
held-out C, no facts: expected False, judge False, rules False/False/False, why: Reason invents facts, omits required checks, and lacks specific details
held-out D, invented score: expected False, judge False, rules False/True/False, why: Agent invented score 0.95 and misstated payee name 'Sablebrook Transit LTD'.
held-out E, no sanctions: expected False, judge False, rules True/False/False, why: Agent did not name the sanctions check result or its details
held-out F, no structuring: expected False, judge False, rules True/False/False, why: Agent's reason does not name the results of the sanctions check and the structuring check.
held-out: judge agrees with my labels: 6 of 6


The judge calibration checks the ruler itself. Six reasons for one case have hand-written labels: A is good, and B to F are bad in different ways. An earlier version of the judge gave one `grounded` verdict for the whole reason. It agreed with the labels only 3 of 6 times, because it passed reasons that named only one check. So the judge now answers each rubric rule separately, and the code computes `grounded` as all rules true. With this change, the judge agrees with the labels 4 of 6 times on the tuning case, and 6 of 6 times on a held-out case that wasn't used for tuning.

In this run, the judge passed two bad reasons: the vague reason B, and reason F, which leaves out the clean sanctions check. In earlier runs, it got 5 of 6 on the tuning case, so the judge isn't fully deterministic, even with a fixed seed.

## Run the agent

In this section, you write the agent loop: the model calls the tools, gets the results, and returns a JSON answer with a decision and a reason. You test the loop with a fake client first, then run prompt v1 on all 39 cases 5 times and report the range of scores.

The API is not fully deterministic, even with `temperature=0` and a fixed seed, so one run is not enough.

In [64]:
TOOL_FUNCTIONS = {
    "screen_name": screen_name,
    "get_customer_activity": get_customer_activity,
}

tool_names = [tool["function"]["name"] for tool in TOOLS]
assert sorted(tool_names) == sorted(TOOL_FUNCTIONS), f"TOOLS {tool_names} vs map {sorted(TOOL_FUNCTIONS)}"
assert all(callable(f) for f in TOOL_FUNCTIONS.values()), "every value must be a function, not a call result"
print("Tool names in TOOLS and in map:", sorted(TOOL_FUNCTIONS))

Tool names in TOOLS and in map: ['get_customer_activity', 'screen_name']


In [65]:
def run_tool(name, arguments_json):
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"unknown tool: {name}"})
    if isinstance(arguments_json, dict):
        args = arguments_json
    else:
        try:
            args = json.loads(arguments_json)
        except json.JSONDecodeError:
            return json.dumps({"error": "arguments are not valid JSON"})
    if not isinstance(args, dict):
        return json.dumps({"error": "arguments must be a JSON object"})
    if not all(isinstance(value, str) for value in args.values()):
        return json.dumps({"error": "arguments must be strings"})
    try:
        result = func(**args)
    except TypeError as e:
        return json.dumps({"error": f"bad arguments: {e}"})
    return json.dumps(result)

got = run_tool("screen_name", '{"name": "Aldona Kureshi"}')
print("run_tool screen_name:", got)

got_dict = run_tool("screen_name", {"name": "Aldona Kureshi"})
assert got_dict == got, f"dict arguments must give the same result: {got_dict}"

bad_dict = run_tool("screen_name", {"name": 123})
assert bad_dict == json.dumps({"error": "arguments must be strings"}), f"bad dict: {bad_dict}"
print("run_tool with dict arguments: 2 checks passed")

run_tool screen_name: [{"id": "SAN-001", "matched_name": "Aldona Kureshi", "score": 1.0}]
run_tool with dict arguments: 2 checks passed


In [66]:
BAD_CALLS = [
    ("screen_names", '{"name": "Aldona Kureshi"}', "unknown tool"),
    ("screen_name", '{"name": "Aldona', "not valid JSON"),
    ("screen_name",  '{"payee": "Aldona Kureshi"}', "bad arguments"),
    ("screen_name", '{"name": null}', "must be strings"),
    ("get_customer_activity", '{"customer_id": 123}', "must be strings"),
    ("screen_name", '["Aldona Kureshi"]', "must be a JSON object"),
]

for name, arguments_json, expected in BAD_CALLS:
    got = json.loads(run_tool(name, arguments_json))
    print(f"run_tool {name} {arguments_json}:", got)
    assert expected in got["error"], f"expected '{expected}', got {got}"

run_tool screen_names {"name": "Aldona Kureshi"}: {'error': 'unknown tool: screen_names'}
run_tool screen_name {"name": "Aldona: {'error': 'arguments are not valid JSON'}
run_tool screen_name {"payee": "Aldona Kureshi"}: {'error': "bad arguments: screen_name() got an unexpected keyword argument 'payee'"}
run_tool screen_name {"name": null}: {'error': 'arguments must be strings'}
run_tool get_customer_activity {"customer_id": 123}: {'error': 'arguments must be strings'}
run_tool screen_name ["Aldona Kureshi"]: {'error': 'arguments must be a JSON object'}


In [67]:
def parse_final(text):
    if not text:
        return {}
    text = text.strip()
    text = text.removeprefix("```json").removeprefix("```")
    text = text.removesuffix("```").strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return {}
    if not isinstance(data, dict):
        return {}
    return data

PARSE_CASES = [
    ('{"decision": "block", "reason": "exact match"}',
     {"decision": "block", "reason": "exact match"}),
    ("I think it should be blocked", {}),
    ('["block"]', {}),
    ('```json\n{"decision": "block", "reason": "reason C"}\n```',
     {"decision": "block", "reason": "reason C"}),
    ('```\n{"decision": "block", "reason": "reason C"}\n```',
     {"decision": "block", "reason": "reason C"}),
    (None, {}),
]

for text, expected in PARSE_CASES:
    got = parse_final(text)
    print(f"parse_final {text!r}:", got)
    assert got == expected, f"expected {expected}, got {got}"

parse_final '{"decision": "block", "reason": "exact match"}': {'decision': 'block', 'reason': 'exact match'}
parse_final 'I think it should be blocked': {}
parse_final '["block"]': {}
parse_final '```json\n{"decision": "block", "reason": "reason C"}\n```': {'decision': 'block', 'reason': 'reason C'}
parse_final '```\n{"decision": "block", "reason": "reason C"}\n```': {'decision': 'block', 'reason': 'reason C'}
parse_final None: {}


In [68]:
AGENT_MODEL = "mistral-small-2603"
MAX_TURNS = 4
AGENT_SEED = 13

def run_case(client, case, prompt, verbose=False):
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": case["request"]},
    ]
    tools_called = set()

    for turn in range(MAX_TURNS):
        response = client.chat.complete(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
            random_seed=AGENT_SEED,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            if verbose:
                print(f"turn {turn} final answer:", msg.content)
            output = parse_final(msg.content)

            if output:
                output["tools_called"] = sorted(tools_called)
            return output

        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            tools_called.add(name)
            arguments_json = tc.function.arguments
            result = run_tool(name, arguments_json)
            if verbose:
                print(f"turn {turn} tool call:", name, arguments_json)
                print(f"turn {turn} tool result:", result[:200])
            messages.append({"role": "tool", "name": name, "content": result, "tool_call_id": tc.id})

    return {}

In [69]:
class FakeAgentClient:
    def __init__(self, replies):
        self.replies = replies
        self.calls = 0
        self.chat = self

    def complete(self, **kwargs):
        last = len(self.replies) - 1
        message = self.replies[min(self.calls, last)]
        self.calls += 1
        choice = SimpleNamespace(message=message)
        return SimpleNamespace(choices=[choice])

def fake_tool_call(name, arguments_json):
    function = SimpleNamespace(name=name, arguments=arguments_json)
    call = SimpleNamespace(id="call-1", function=function)
    return SimpleNamespace(content="", tool_calls=[call])

def fake_answer(text):
    return SimpleNamespace(content=text, tool_calls=None)

In [70]:
GOOD = '{"decision": "block", "reason": "exact match"}'
GOOD_DICT = {"decision": "block", "reason": "exact match"}
GOOD_SCREEN = {"decision": "block", "reason": "exact match", "tools_called": ["screen_name"]}
GOOD_BAD_NAME = {"decision": "block", "reason": "exact match", "tools_called": ["screen_name", "screen_names"]}
SCREEN = fake_tool_call("screen_name", '{"name": "Aldona Kureshi"}')
BAD_NAME = fake_tool_call("screen_names", '{"name": "Aldona Kureshi"}')

LOOP_CASES = [
    ([SCREEN, fake_answer(GOOD)], GOOD_SCREEN, 2),
    ([SCREEN], {}, 4),
    ([SCREEN, fake_answer("```json\n" + GOOD + "\n```")], GOOD_SCREEN, 2),
    ([BAD_NAME, SCREEN, fake_answer(GOOD)], GOOD_BAD_NAME, 3),
]

for replies, expected, expected_calls in LOOP_CASES:
    fake_client = FakeAgentClient(replies)
    got = run_case(fake_client, CASES[0], "test prompt")
    print(f"run_case, {fake_client.calls} calls:", got)
    assert got == expected, f"expected {expected}, got {got}"
    assert fake_client.calls == expected_calls, f"expected {expected_calls} calls, got {fake_client.calls}"

run_case, 2 calls: {'decision': 'block', 'reason': 'exact match', 'tools_called': ['screen_name']}
run_case, 4 calls: {}
run_case, 2 calls: {'decision': 'block', 'reason': 'exact match', 'tools_called': ['screen_name']}
run_case, 3 calls: {'decision': 'block', 'reason': 'exact match', 'tools_called': ['screen_name', 'screen_names']}


In [71]:
PROMPT_V1 = """You are doing a compliance check of a payment.
You must check the payee and the customer activity using tools.
There are three decisions: block, escalate, clear.
Reply only in JSON like: {"decision": "...", "reason": "..."}
"""

The next cell runs one real case with `verbose=True`, so you can see what the agent does: which tools it calls, with which arguments, and what it answers. The model can call both tools in one turn. Each tool result goes back to the model with the `tool_call_id` of its call, so the model knows which result answers which call. The output shows only the first 200 characters of each tool result.

In [72]:
real_case = CASES[0]
print("case:", real_case["id"], real_case["request"])
got = run_case(client, real_case, PROMPT_V1, verbose=True)
print("run_case result:", got)
print("expected decision:", real_case["expected_decision"])

case: case-01 Customer C-04 wants to send €2,500 to Aldona Qureshi.
turn 0 tool call: screen_name {"name": "Aldona Qureshi"}
turn 0 tool result: [{"id": "SAN-001", "matched_name": "Aldona Qureshi", "score": 1.0}]
turn 0 tool call: get_customer_activity {"customer_id": "C-04"}
turn 0 tool result: {"customer_id": "C-04", "as_of": "2026-11-01", "transactions": [{"amount": 180.26, "currency": "EUR", "date": "2026-09-06", "id": "C-04-T001", "type": "cash_out"}, {"amount": 233.97, "currency": "EUR"
turn 1 final answer: {"decision": "block", "reason": "The payee 'Aldona Qureshi' is on the sanctions list with a score of 1.0."}
run_case result: {'decision': 'block', 'reason': "The payee 'Aldona Qureshi' is on the sanctions list with a score of 1.0.", 'tools_called': ['get_customer_activity', 'screen_name']}
expected decision: block


In [73]:
def run_all(client, prompt):
    outputs = {}
    for case in load_cases():
        outputs[case["id"]] = run_case(client, case, prompt)
    return outputs

fake_client = FakeAgentClient([fake_answer(GOOD)])
fake_outputs = run_all(fake_client, "test prompt")
print("run_all with fake:", len(fake_outputs), "outputs,", fake_client.calls, "calls")
assert len(fake_outputs) == len(CASES), f"outputs: {len(fake_outputs)}"
assert fake_client.calls == len(CASES), f"calls: {fake_client.calls}"
assert fake_outputs["case-01"] == {**GOOD_DICT, "tools_called": []}, f"case-01: {fake_outputs['case-01']}"

run_all with fake: 39 outputs, 39 calls


In [74]:
BOTH_TOOLS = {"screen_name", "get_customer_activity"}

def answers_without_both_tools(outputs):
    return [
        case_id for case_id, output in outputs.items()
        if output and not BOTH_TOOLS.issubset(output["tools_called"])
    ]

In [75]:
TOOLS_TEST_OUTPUTS = {
    "a": {"decision": "block", "reason": "x", "tools_called": ["get_customer_activity", "screen_name"]},
    "b": {"decision": "block", "reason": "x", "tools_called": ["screen_name"]},
    "c": {},
    "d": {"decision": "clear", "reason": "x", "tools_called": []},
}
got = answers_without_both_tools(TOOLS_TEST_OUTPUTS)
print("answers without both tools (expect ['b', 'd']):", got)
assert got == ["b", "d"], f"got {got}"

answers without both tools (expect ['b', 'd']): ['b', 'd']


In [76]:
shutil.rmtree(judge_module.CACHE_DIR, ignore_errors=True)  # every full run starts with an empty judge cache
assert not judge_module.CACHE_DIR.exists(), "judge cache not cleared"

V1_RUNS = 5

outputs_runs_v1 = []
scores_runs_v1 = []
for run in range(V1_RUNS):
    outputs = run_all(client, PROMPT_V1)
    empty = [case_id for case_id, output in outputs.items() if not output]
    scores = evaluate(client, outputs)
    outputs_runs_v1.append(outputs)
    scores_runs_v1.append(scores)
    print(f"v1 run {run}: total", scores["total"], "empty:", empty)

v1 run 0: total {'passed': 5, 'n': 39, 'rate': 0.13} empty: []
v1 run 1: total {'passed': 4, 'n': 39, 'rate': 0.1} empty: []
v1 run 2: total {'passed': 6, 'n': 39, 'rate': 0.15} empty: []
v1 run 3: total {'passed': 3, 'n': 39, 'rate': 0.08} empty: []
v1 run 4: total {'passed': 5, 'n': 39, 'rate': 0.13} empty: []


In [77]:
for group in ["total", "easy", "hard"]:
    rates = [s[group]["rate"] for s in scores_runs_v1]
    print(f"v1 {group} rate range:", min(rates), "-", max(rates))

v1 total rate range: 0.08 - 0.15
v1 easy rate range: 0.2 - 0.33
v1 hard rate range: 0.0 - 0.08


The five runs give different scores for the same prompt. In this run, the total pass rate is between 0.08 and 0.15. If you report only the best run, that's one more way to inflate the score. So the notebook reports the range, and the comparison with prompt v2 uses it.

## Inflate the score

In this section, you play someone who wants a better number without a better agent. Trick 1 cuts the rubric to only one rule. Trick 2 removes the hard cases from the dataset. Both tricks rescore the same v1 answers, so the agent doesn't change, but the score goes up. Each trick restores the original file at the end.

In [78]:
original_rubric = judge_module.RUBRIC_PATH.read_text()

RELAXED = [
    ("all three rules hold", "this rule holds"),
    ("For rules 2 and 3, look only at the agent's reason, not at the evidence.\n", ""),
    ("2. The reason names the result of both checks, the sanctions check of the payee and the structuring check of the customer, even a clean one.\n", ""),
    ("3. Each result includes the specific details the evidence gives for it: numbers, ranges, and thresholds.\n", ""),
    ('"rule_2": true or false, "rule_3": true or false, ', ""),
    ("only if all three rules are true", "only if this rule is true"),
]

relaxed_rubric = original_rubric
for old, new in RELAXED:
    assert old in relaxed_rubric, f"not found in rubric: {old!r}"
    relaxed_rubric = relaxed_rubric.replace(old, new)

assert judge_module.rule_names(relaxed_rubric) == ["rule_1"], "relaxed rubric must keep only rule 1"
print("relaxed rubric:")
print(relaxed_rubric)

relaxed rubric:
# Judge rubric

You check one answer from a compliance agent.
You get: the request, the evidence, the agent's decision, and the agent's reason.
The decision is already correct. You check only the reason.

The reason is grounded only if this rule holds:
1. Every fact in the reason matches the evidence. Nothing is invented.

Check each rule separately. Answer with JSON only:
{"rule_1": true or false, "grounded": true only if this rule is true, "why": "one short sentence"}



In [79]:
try:
    judge_module.RUBRIC_PATH.write_text(relaxed_rubric)
    relaxed_scores_v1 = [evaluate(client, outputs) for outputs in outputs_runs_v1]
finally:
    judge_module.RUBRIC_PATH.write_text(original_rubric)

assert judge_module.RUBRIC_PATH.read_text() == original_rubric, "rubric not restored"

for group in ["total", "easy", "hard"]:
    before = [s[group]["rate"] for s in scores_runs_v1]
    after = [s[group]["rate"] for s in relaxed_scores_v1]
    print(f"trick 1, {group}: before", min(before), "-", max(before), "after", min(after), "-", max(after))

for run in range(V1_RUNS):
    before = scores_runs_v1[run]["total"]["passed"]
    after = relaxed_scores_v1[run]["total"]["passed"]
    print(f"trick 1, run {run}: passed {before} -> {after} of {len(CASES)}")

before_sum = sum([s["total"]["passed"] for s in scores_runs_v1])
after_sum = sum([s["total"]["passed"] for s in relaxed_scores_v1])
print(f"trick 1, all runs: passed {before_sum} -> {after_sum}")
assert after_sum > before_sum, "trick 1 did not raise passed over all runs"

trick 1, total: before 0.08 - 0.15 after 0.41 - 0.49
trick 1, easy: before 0.2 - 0.33 after 0.6 - 0.73
trick 1, hard: before 0.0 - 0.08 after 0.29 - 0.38
trick 1, run 0: passed 5 -> 19 of 39
trick 1, run 1: passed 4 -> 18 of 39
trick 1, run 2: passed 6 -> 17 of 39
trick 1, run 3: passed 3 -> 16 of 39
trick 1, run 4: passed 5 -> 18 of 39
trick 1, all runs: passed 23 -> 88


Trick 1 has a big effect on the same answers: the total pass rate goes from 0.08-0.15 to 0.41-0.49. Most v1 reasons fail rule 2 or rule 3: they don't name both checks, or they don't give the numbers. The relaxed rubric keeps only rule 1, so almost every answer with a correct decision now passes. The score still can't go above the number of correct decisions (18-20 of 39), because the code checks the decision before the judge sees the reason.

In [80]:
for run, outputs in enumerate(outputs_runs_v1):
    correct = [case["id"] for case in CASES if outputs[case["id"]].get("decision") == case["expected_decision"]]
    print(f"v1 run {run}: correct decisions", len(correct), "of", len(CASES))

v1 run 0: correct decisions 20 of 39
v1 run 1: correct decisions 19 of 39
v1 run 2: correct decisions 18 of 39
v1 run 3: correct decisions 18 of 39
v1 run 4: correct decisions 19 of 39


In [81]:
CASES_FILE = DATA_DIR / "cases.jsonl"
original_cases = CASES_FILE.read_text()
all_lines = original_cases.splitlines()
assert len(all_lines) == len(CASES), f"lines in file: {len(all_lines)}"

kept_lines = [line for line in all_lines if json.loads(line)["difficulty"] == "easy"]
easy_only_cases = "\n".join(kept_lines) + "\n"

n_easy = len([case for case in CASES if case["difficulty"] == "easy"])
assert len(kept_lines) == n_easy, f"kept: {len(kept_lines)}"
print("trick 2 dataset:", len(kept_lines), "of", len(all_lines), "cases kept")

trick 2 dataset: 15 of 39 cases kept


In [82]:
try:
    CASES_FILE.write_text(easy_only_cases)
    dropped_scores_v1 = [evaluate(client, outputs) for outputs in outputs_runs_v1]
finally:
    CASES_FILE.write_text(original_cases)

assert CASES_FILE.read_text() == original_cases, "cases not restored"

for group in ["total", "easy"]:
    before = [s[group]["rate"] for s in scores_runs_v1]
    after = [s[group]["rate"] for s in dropped_scores_v1]
    print(f"trick 2, {group}: before", min(before), "-", max(before), "after", min(after), "-", max(after))
print("trick 2, hard after (run 0):", dropped_scores_v1[0]["hard"])

for run in range(V1_RUNS):
    before = scores_runs_v1[run]["total"]
    after = dropped_scores_v1[run]["total"]
    print(f"trick 2, run {run}: passed {before['passed']} of {before['n']} -> {after['passed']} of {after['n']}, rate {before['rate']} -> {after['rate']}")
    assert after["n"] == n_easy, f"trick 2, run {run}: total n is {after['n']}, not {n_easy}"
    assert dropped_scores_v1[run]["hard"]["n"] == 0, f"trick 2, run {run}: hard cases are still in the dataset"

before_passed = sum([s["total"]["passed"] for s in scores_runs_v1])
before_n = sum([s["total"]["n"] for s in scores_runs_v1])
after_passed = sum([s["total"]["passed"] for s in dropped_scores_v1])
after_n = sum([s["total"]["n"] for s in dropped_scores_v1])
print(f"trick 2, all runs: passed {before_passed} of {before_n} -> {after_passed} of {after_n}")
assert after_passed / after_n > before_passed / before_n, "trick 2 did not raise the rate over all runs"

trick 2, total: before 0.08 - 0.15 after 0.2 - 0.33
trick 2, easy: before 0.2 - 0.33 after 0.2 - 0.33
trick 2, hard after (run 0): {'passed': 0, 'n': 0, 'rate': None}
trick 2, run 0: passed 5 of 39 -> 5 of 15, rate 0.13 -> 0.33
trick 2, run 1: passed 4 of 39 -> 3 of 15, rate 0.1 -> 0.2
trick 2, run 2: passed 6 of 39 -> 4 of 15, rate 0.15 -> 0.27
trick 2, run 3: passed 3 of 39 -> 3 of 15, rate 0.08 -> 0.2
trick 2, run 4: passed 5 of 39 -> 5 of 15, rate 0.13 -> 0.33
trick 2, all runs: passed 23 of 195 -> 20 of 75


Trick 2 has a smaller effect: the total pass rate goes from 0.08-0.15 to 0.20-0.33. After the trick, the total rate is just the easy rate, and v1 fails most easy cases too, so there isn't much to gain. The number of passed cases doesn't go up in any run, for example 6 of 39 before and 4 of 15 after. Only the counts show what happened: the dataset has 15 cases instead of 39, and the hard group is 0 of 0, with rate `None`.

## Freeze the scorer with oathgate

In this section, you write an oathgate spec that lists every file that decides the score, freeze it, and record a prediction for prompt v2 before you run it. Then you repeat both tricks with `oathgate check` after each one.

In [83]:
%%writefile compliance_eval/oathgate.toml
[dataset]
path = "data/cases.jsonl"

[metrics.pass_rate]
impl = "pass_rate.py"
extra_files = ["judge.py", "judge_rubric.md", "data/world.json"]

Overwriting compliance_eval/oathgate.toml


In [84]:
OATHGATE_SPEC = "compliance_eval/oathgate.toml"
assert shutil.which("oathgate"), "oathgate not found: pip install oathgate==0.1.0"

def oathgate(*args):
    result = subprocess.run(["oathgate", *args, "--spec", OATHGATE_SPEC], capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()

code, out = oathgate("freeze")
print("oathgate freeze:", code, out)
assert code == 0, out

oathgate freeze: 0 eaa208302581c168b451800ab296b8e3f666610accd87be82f2e9820b3cd21ba


In [85]:
PREDICTION = 0.60
LOCK_PATH = Path(OATHGATE_SPEC).parent / "oathgate.lock"

if not LOCK_PATH.exists():
    code, out = oathgate("predict", "--metric", "pass_rate", "--at-least", str(PREDICTION))
    print("oathgate predict:", code, out)
    assert code == 0, out

prediction = json.loads(LOCK_PATH.read_text())["predictions"][0]
print("prediction in lock:", prediction)
expected_prediction = {"metric": "pass_rate", "operator": ">=", "value": PREDICTION}
assert prediction == expected_prediction, f"lock says {prediction}, code says {PREDICTION}: delete the old lock on purpose"

prediction in lock: {'metric': 'pass_rate', 'operator': '>=', 'value': 0.6}


In [86]:
checks = []

try:
    judge_module.RUBRIC_PATH.write_text(relaxed_rubric)
    code, out = oathgate("check")
    print("check with trick 1 (relaxed rubric):", code, out.splitlines()[0])
    checks.append(code)
finally:
    judge_module.RUBRIC_PATH.write_text(original_rubric)

try:
    CASES_FILE.write_text(easy_only_cases)
    code, out = oathgate("check")
    print("check with trick 2 (no hard cases):", code, out.splitlines()[0])
    checks.append(code)
finally:
    CASES_FILE.write_text(original_cases)

code, out = oathgate("check")
print("check after restore:", code, out.splitlines()[0])
checks.append(code)

assert checks == [1, 1, 0], f"checks: {checks}"

check with trick 1 (relaxed rubric): 1 ruler moved
check with trick 2 (no hard cases): 1 ruler moved
check after restore: 0 ok eaa208302581c168b451800ab296b8e3f666610accd87be82f2e9820b3cd21ba


`oathgate check` returns `1` for both tricks, the big one and the small one, because it doesn't look at the score. It only compares the hash of the files with the frozen hash. After the files are restored, it returns `0` again. Note that oathgate checks that the scorer didn't change, not that the scorer is correct. That's why the scorer has its own tests before the freeze.

## Improve the prompt

In this section, you write prompt v2 with the full policy: the thresholds, the structuring rule, and the rule that the strictest decision wins. The `report` function first runs `oathgate check` and raises if the scorer moved. Then it compares the lowest total score of five v2 runs with the prediction. If the prediction is missed, the notebook shows it instead of hiding it.

In [87]:
def report(label, scores_runs):
    code, out = oathgate("check")
    if code != 0:
        raise RuntimeError(f"oathgate check failed, code {code}: {out}")

    totals = [s["total"]["rate"] for s in scores_runs]
    low, high = min(totals), max(totals)
    met = PREDICTION <= low
    print(f"{label}: total {low} - {high}, prediction >= {PREDICTION}, met: {met}")
    return met

In [88]:
def fake_scores(rates):
    return [{"total": {"rate": rate}} for rate in rates]

REPORT_CASES = [
    ([0.62, 0.54, 0.64, 0.59, 0.62], False),
    ([0.65, 0.70, 0.68, 0.66, 0.72], True),
    ([0.60, 0.64, 0.62, 0.61, 0.63], True),
]

for rates, expected in REPORT_CASES:
    got = report("report test", fake_scores(rates))
    assert got == expected, f"rates {rates}: got {got}, expected {expected}"

raised = False
try:
    judge_module.RUBRIC_PATH.write_text(relaxed_rubric)
    report("report test, trick 1", fake_scores([0.9, 0.9, 0.9, 0.9, 0.9]))
except RuntimeError as e:
    raised = True
    print("report with trick 1 raised:", str(e).splitlines()[0])
finally:
    judge_module.RUBRIC_PATH.write_text(original_rubric)

assert judge_module.RUBRIC_PATH.read_text() == original_rubric, "rubric not restored"
assert raised, "report did not raise with trick 1"

report test: total 0.54 - 0.64, prediction >= 0.6, met: False
report test: total 0.65 - 0.72, prediction >= 0.6, met: True
report test: total 0.6 - 0.64, prediction >= 0.6, met: True
report with trick 1 raised: oathgate check failed, code 1: ruler moved


In [89]:
PROMPT_V2 = """You are doing a compliance check of a payment.
You must check the payee and the customer activity using tools.
There are three decisions: block, escalate, clear.
There are conditions: block when the highest match score from screen_name is >= 0.92, escalate by sanctions when it is >= 0.80 and < 0.92.
Clear by sanctions when the score is less than 0.80, or when there is no match.
Structuring means: 3+ cash_in with amount between 9000 and 9999.99 within 29 days.
When structuring is found then escalate.
When you receive two signals return the most significant, for example: block and escalate leads to block, and escalate and clear leads to escalate.
Always include results for both checks including numbers.
Reply only in JSON like: {"decision": "...", "reason": "..."}
"""

In [90]:
code, out = oathgate("check")
if code != 0:
    raise RuntimeError(f"oathgate check failed before v2, code {code}: {out}")
print("oathgate check before v2 runs:", code, out.splitlines()[0])

V2_RUNS = V1_RUNS
outputs_runs_v2 = []
scores_runs_v2 = []
for run in range(V2_RUNS):
    outputs = run_all(client, PROMPT_V2)
    empty = [case_id for case_id, output in outputs.items() if not output]
    scores = evaluate(client, outputs)
    outputs_runs_v2.append(outputs)
    scores_runs_v2.append(scores)
    print(f"v2 run {run}: total", scores["total"], "empty:", empty)

for group in ["total", "easy", "hard"]:
    rates = [s[group]["rate"] for s in scores_runs_v2]
    print(f"v2 {group} rate range:", min(rates), "-", max(rates))

met_v2 = report("prompt v2", scores_runs_v2)

oathgate check before v2 runs: 0 ok eaa208302581c168b451800ab296b8e3f666610accd87be82f2e9820b3cd21ba
v2 run 0: total {'passed': 26, 'n': 39, 'rate': 0.67} empty: []
v2 run 1: total {'passed': 25, 'n': 39, 'rate': 0.64} empty: []
v2 run 2: total {'passed': 27, 'n': 39, 'rate': 0.69} empty: []
v2 run 3: total {'passed': 26, 'n': 39, 'rate': 0.67} empty: []
v2 run 4: total {'passed': 26, 'n': 39, 'rate': 0.67} empty: []
v2 total rate range: 0.64 - 0.69
v2 easy rate range: 0.8 - 0.87
v2 hard rate range: 0.5 - 0.58
prompt v2: total 0.64 - 0.69, prediction >= 0.6, met: True


In [91]:
for run, outputs in enumerate(outputs_runs_v2):
    correct = [case["id"] for case in CASES if outputs[case["id"]].get("decision") == case["expected_decision"]]
    passed = scores_runs_v2[run]["total"]["passed"]
    print(f"v2 run {run}: correct decisions {len(correct)} of {len(CASES)}, passed {passed}")

v2 run 0: correct decisions 34 of 39, passed 26
v2 run 1: correct decisions 34 of 39, passed 25
v2 run 2: correct decisions 35 of 39, passed 27
v2 run 3: correct decisions 35 of 39, passed 26
v2 run 4: correct decisions 35 of 39, passed 26


In [92]:
for label, runs in [("v1", outputs_runs_v1), ("v2", outputs_runs_v2)]:
    for run, outputs in enumerate(runs):
        missing = answers_without_both_tools(outputs)
        print(f"{label} run {run}: answers without both tools: {len(missing)} of {len(outputs)}", missing)

v1 run 0: answers without both tools: 0 of 39 []
v1 run 1: answers without both tools: 0 of 39 []
v1 run 2: answers without both tools: 0 of 39 []
v1 run 3: answers without both tools: 0 of 39 []
v1 run 4: answers without both tools: 0 of 39 []
v2 run 0: answers without both tools: 0 of 39 []
v2 run 1: answers without both tools: 0 of 39 []
v2 run 2: answers without both tools: 0 of 39 []
v2 run 3: answers without both tools: 0 of 39 []
v2 run 4: answers without both tools: 0 of 39 []


Prompt v2 states the full policy, and the total pass rate goes from 0.08-0.15 to 0.64-0.69. The two ranges don't overlap, so the difference is bigger than the run-to-run noise. Most of the noise is in the reasons, not in the decisions: v2 gets 34 or 35 of 39 decisions right in every run, but the number of passed cases moves between 25 and 27. The scorer is the same as before: `report` runs `oathgate check` first. During development, the lowest v2 score was sometimes below 0.60 and sometimes above it, so v2 is close to the predicted line. In this run, the lowest v2 score is 0.64, so the prediction `>= 0.60` is met.

## Limitations

This notebook shows the method on a small synthetic example. Keep these limits in mind:

- The judge uses the same model as the agent, `mistral-small-2603`. The judge calibration checks that the judge agrees with hand-written labels, but it doesn't test whether the judge favors answers from its own model.
- The judge was calibrated on only 12 reasons for 2 cases from the same dataset. The reasons are written by hand, not taken from the agent's answers. In this run, the judge agreed with the labels 10 of 12 times. It passed the vague reason (variant B) and the reason that left out the clean sanctions check (variant F). Its `why` text is sometimes wrong even when the verdict is right.
- The thresholds and the structuring rule are invented for this demo. They aren't real compliance standards.
- The structuring window means that the first and the last deposit are at most 29 days apart.
- Five runs per prompt are few. Ranges that don't overlap show that the difference is bigger than the noise in these runs, but this isn't a statistical test.
- The scorer was changed after the first results. An external review and the judge calibration found two bugs in the ruler: the structuring evidence listed too few deposits, and the judge didn't apply rule 2. Each time, the scorer was frozen again, and the same prediction, `>= 0.60`, was recorded before the new v2 runs. The numbers in this notebook come from one final run.
- In a real project, commit the lock file before the evaluation. This notebook creates its own lock when it runs, so you can run it from an empty folder.
- The judge cache (`judge_cache/`) isn't part of the frozen files. The notebook deletes it before the first agent run, so every full run grades all answers again. If you keep the cache between runs, anyone who edits it can change the score, and `oathgate check` won't notice.
- Tool use isn't part of the pass rate. The notebook only reports how many answers didn't call both tools: 0 of 39 in every run.

## Summary

In this cookbook, you measure a compliance agent with a pass rate and see two ways to inflate the score without a better agent: a relaxed rubric and a dataset without the hard cases. After you freeze the scorer with oathgate, `oathgate check` catches both tricks. Then you improve the prompt, and the score goes up with the same scorer.

### What you built

The notebook creates every part itself, so you can run it from an empty folder:

- A synthetic world: a sanctions list and customers with 60 days of transactions
- An agent with two tools, `screen_name` and `get_customer_activity`
- 39 test cases labeled by the policy as code, split into easy and hard
- A scorer: the decision is checked by code, and the reason by an LLM judge
- An oathgate spec that freezes the rubric, the judge, the scoring code, and the data

### Mistral features used

The notebook uses these Mistral API features:

- Tool calling: the agent calls the tools and gets the results back with `tool_call_id`
- JSON mode for the judge: `response_format={"type": "json_object"}`
- `temperature=0` and a fixed `random_seed` for more consistent runs
- A retry config in the client for rate limit (`429`) errors, server errors, and connection timeouts

### Other services

The notebook also uses one open-source tool:

- [oathgate](https://pypi.org/project/oathgate/): freezes the scorer and checks that it didn't change

To learn more about tool calling with Mistral, see the [function calling guide](https://docs.mistral.ai/capabilities/function_calling/).